# 02 — Data Exploration
Deeper look at the metadata (demographics, CDR/MMSE distributions by class) and a data-pipeline
integrity check (how many CSV rows have a matching image, per split and per class).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config as cfg

train_df = pd.read_csv(cfg.TRAIN_CSV)
test_df = pd.read_csv(cfg.TEST_CSV)
train_df.head()

In [ ]:
# Class balance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_df['class'].value_counts().reindex(cfg.CLASS_NAMES).plot.bar(ax=axes[0], title='Train class counts')
test_df['class'].value_counts().reindex(cfg.CLASS_NAMES).plot.bar(ax=axes[1], title='Test class counts')
plt.tight_layout()
plt.show()

print("Note: OASIS is imbalanced (NonDemented dominates). We correct for this with class_weight "
      "during training (see src/data_loader.py get_train_val_test_datasets).")

In [ ]:
# CDR (Clinical Dementia Rating) and MMSE by class — sanity-check that the class labels
# line up with the clinical scores as expected (CDR should increase with severity)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=train_df, x='class', y='CDR', order=cfg.CLASS_NAMES, ax=axes[0])
axes[0].set_title('CDR by class')
axes[0].tick_params(axis='x', rotation=30)

sns.boxplot(data=train_df, x='class', y='MMSE', order=cfg.CLASS_NAMES, ax=axes[1])
axes[1].set_title('MMSE by class')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Image availability check (cell 3 referenced from data/README_DATA.md)
Confirms how many CSV rows actually resolve to an image file before you spend time training.

In [ ]:
from src.data_loader import get_filepaths_and_labels
from collections import Counter

for split in ['train', 'test']:
    paths, labels = get_filepaths_and_labels(split)
    counts = Counter(labels)
    print(f"\n[{split}] images found per class:")
    for i, name in enumerate(cfg.CLASS_NAMES):
        print(f"  {name:20s}: {counts.get(i, 0)}")